# satlens — training

Fine-tunes SegFormer-b0 on OpenEarthMap. Run top to bottom on a T4 GPU (`Runtime → Change runtime type → T4 GPU`). Takes ~45 min.


## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets torch torchvision huggingface_hub Pillow tqdm matplotlib

## 2. Imports

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    get_scheduler,
)
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 3. Download OpenEarthMap dataset

In [ ]:

from datasets import load_dataset

print('Downloading OpenEarthMap dataset (this may take a few minutes)...')
dataset = load_dataset('open-earth-map/open-earth-map', trust_remote_code=True)
print(dataset)
print(f"\nTrain samples: {len(dataset['train'])}")
print(f"Val samples:   {len(dataset['val'])}")

## 4. Explore the data

In [ ]:
CLASSES = [
    'Background',
    'Bareland',
    'Rangeland',
    'Developed space',
    'Road',
    'Tree',
    'Water',
    'Agriculture land',
    'Building',
]

PALETTE = np.array([
    [0,   0,   0],
    [128, 0,   0],
    [0,   128, 0],
    [128, 128, 0],
    [255, 255, 0],
    [0,   64,  0],
    [0,   0,   255],
    [0,   255, 128],
    [255, 0,   0],
], dtype=np.uint8)

NUM_CLASSES = len(CLASSES)

def colorize_mask(mask):
    return Image.fromarray(PALETTE[mask])

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
for i, ax_row in enumerate(axes):
    sample = dataset['train'][i]
    img  = sample['image']
    mask = np.array(sample['label'])
    ax_row[0].imshow(img)
    ax_row[0].set_title(f'Sample {i} — Image')
    ax_row[0].axis('off')
    ax_row[1].imshow(colorize_mask(mask))
    ax_row[1].set_title(f'Sample {i} — Label')
    ax_row[1].axis('off')

patches = [mpatches.Patch(color=PALETTE[i]/255, label=CLASSES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

## 5. Dataset class

In [ ]:
processor = SegformerImageProcessor(
    do_resize=True,
    size={'height': 512, 'width': 512},
    do_normalize=True,
)

class OpenEarthMapDataset(Dataset):
    def __init__(self, hf_split, processor, augment=False):
        self.data      = hf_split
        self.processor = processor
        self.augment   = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image  = sample['image'].convert('RGB')
        mask   = sample['label']  # PIL Image, single-channel

        if self.augment:
            # Random horizontal flip
            if torch.rand(1) > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                mask  = mask.transpose(Image.FLIP_LEFT_RIGHT)
            # Random vertical flip
            if torch.rand(1) > 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                mask  = mask.transpose(Image.FLIP_TOP_BOTTOM)

        encoded = self.processor(
            images=image,
            segmentation_maps=mask,
            return_tensors='pt',
        )

        return {
            'pixel_values': encoded['pixel_values'].squeeze(0),
            'labels':       encoded['labels'].squeeze(0),
        }

train_dataset = OpenEarthMapDataset(dataset['train'], processor, augment=True)
val_dataset   = OpenEarthMapDataset(dataset['val'],   processor, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)

print(f"{len(train_loader)} train / {len(val_loader)} val batches")

## 6. Load pretrained SegFormer-b0

In [ ]:

id2label = {i: c for i, c in enumerate(CLASSES)}
label2id = {c: i for i, c in id2label.items()}

model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/mit-b0',
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # new head for our class count
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model parameters: {total_params:.1f}M')

## 7. Training setup

In [ ]:
NUM_EPOCHS    = 20
LR            = 6e-5
WARMUP_STEPS  = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler   = get_scheduler(
    'cosine',
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

def mean_iou(preds, labels, num_classes, ignore_index=0):
    """Compute mean Intersection-over-Union across classes."""
    ious = []
    preds  = preds.flatten()
    labels = labels.flatten()
    for cls in range(1, num_classes):  # skip background
        pred_cls  = preds  == cls
        label_cls = labels == cls
        intersection = (pred_cls & label_cls).sum().item()
        union        = (pred_cls | label_cls).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0

print(f'Training for {NUM_EPOCHS} epochs ({total_steps} total steps)')

## 8. Train

In [ ]:
best_miou       = 0.0
train_losses    = []
val_mious       = []

for epoch in range(NUM_EPOCHS):
        model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]')

    for batch in pbar:
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

        model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [val]'):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels       = batch['labels']

            outputs = model(pixel_values=pixel_values)
            logits  = outputs.logits  # (B, C, H/4, W/4)

            upsampled = torch.nn.functional.interpolate(
                logits, size=labels.shape[-2:], mode='bilinear', align_corners=False
            )
            preds = upsampled.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)

    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    miou       = mean_iou(all_preds, all_labels, NUM_CLASSES)
    val_mious.append(miou)

    print(f'Epoch {epoch+1} | loss: {avg_loss:.4f} | val mIoU: {miou:.4f}')

        if miou > best_miou:
        best_miou = miou
        model.save_pretrained('/content/satlens-segformer-best')
        processor.save_pretrained('/content/satlens-segformer-best')
        print(f'  ✅ New best model saved (mIoU: {best_miou:.4f})')

print(f'\nTraining complete. Best mIoU: {best_miou:.4f}')

## 9. Plot training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, marker='o')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(val_mious, marker='o', color='green')
ax2.set_title('Validation mIoU')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('mIoU')
ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print(f'Best validation mIoU: {best_miou:.4f}')

## 10. Visualize predictions

In [ ]:
model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Predictions on validation set', fontsize=14)

for i, ax_row in enumerate(axes):
    sample       = dataset['val'][i]
    image        = sample['image'].convert('RGB')
    true_mask    = np.array(sample['label'])

    encoded = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**encoded)
    logits = outputs.logits
    upsampled = torch.nn.functional.interpolate(
        logits, size=(512, 512), mode='bilinear', align_corners=False
    )
    pred_mask = upsampled.argmax(dim=1).squeeze(0).cpu().numpy()

    ax_row[0].imshow(image.resize((512, 512)))
    ax_row[0].set_title('Image')
    ax_row[0].axis('off')

    ax_row[1].imshow(PALETTE[true_mask])
    ax_row[1].set_title('Ground truth')
    ax_row[1].axis('off')

    ax_row[2].imshow(PALETTE[pred_mask])
    ax_row[2].set_title('Predicted')
    ax_row[2].axis('off')

patches = [mpatches.Patch(color=PALETTE[i]/255, label=CLASSES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig('/content/predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Upload model to Hugging Face Hub

This step saves the model to your HF account so the Gradio app can load it.

You'll need a **write** token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import HfApi, login

HF_USERNAME  = 'YOUR_HF_USERNAME'   # ← change this
REPO_NAME    = 'satlens-segformer'
REPO_ID      = f'{HF_USERNAME}/{REPO_NAME}'

login()  # will prompt for your token

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)

model.push_to_hub(REPO_ID)
processor.push_to_hub(REPO_ID)

print(f'\n✅ Model uploaded to: https://huggingface.co/{REPO_ID}')
print(f'\nSet MODEL_ID={REPO_ID} in your Hugging Face Space environment variables.')